# L08B follow-along: factor, preserve, reuse

This notebook follows the lecture's three most reusable CNN design moves:

1. **Factor** a standard convolution into depthwise spatial filtering and pointwise channel mixing.
2. **Preserve** an identity path and learn only a residual correction.
3. **Reuse** a backbone under explicit weight/state contracts, then stage a validation-triggered partial unfreeze.

For each part: **predict → calculate → run → inspect → explain**. All tensors are synthetic and seeded; there are no downloads.

## 0 · Setup

Before running the next cell, predict which number will be larger: the parameter reduction from a depthwise-separable convolution, or the ratio $C_{out}/k^2$? We will derive the exact answer.

In [1]:
import copy
import math
import random

import numpy as np
import torch
from torch import nn

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)


def n_params(module, *, trainable_only=False):
    params = module.parameters()
    if trainable_only:
        params = (p for p in params if p.requires_grad)
    return sum(p.numel() for p in params)

print(f"torch={torch.__version__}; seed={SEED}")

torch=2.13.0; seed=8


## 1 · Factor one expensive convolution

We use exactly the lecture example:

- input feature map: $28\times28\times64$;
- output feature map: $28\times28\times128$;
- kernel: $3\times3$;
- bias omitted.

**Predict before running:**

- Standard weights: $3\cdot3\cdot64\cdot128 = \;?$  
- Depthwise weights: $3\cdot3\cdot64 = \;?$  
- Pointwise weights: $64\cdot128 = \;?$  
- Will both implementations return the same output *shape*?

In [2]:
H = W = 28
k = 3
c_in = 64
c_out = 128

standard_weights = k * k * c_in * c_out
depthwise_weights = k * k * c_in
pointwise_weights = c_in * c_out
separable_weights = depthwise_weights + pointwise_weights

standard_macs = H * W * standard_weights
separable_macs = H * W * separable_weights

print(f"standard weights : {standard_weights:>10,}")
print(f"depthwise weights: {depthwise_weights:>10,}")
print(f"pointwise weights: {pointwise_weights:>10,}")
print(f"separable total  : {separable_weights:>10,}")
print(f"parameter ratio : {standard_weights / separable_weights:>10.3f}×")
print(f"standard MACs   : {standard_macs:>10,}")
print(f"separable MACs  : {separable_macs:>10,}")

assert (standard_weights, separable_weights) == (73_728, 8_768)
assert (standard_macs, separable_macs) == (57_802_752, 6_874_112)

standard weights :     73,728
depthwise weights:        576
pointwise weights:      8,192
separable total  :      8,768
parameter ratio :      8.409×
standard MACs   : 57,802,752
separable MACs  :  6,874,112


### Run the two node paths

A standard convolution mixes space and channels in one operation. The separable path exposes two nodes:

$$x\;\longrightarrow\;\text{depthwise }3\!\times\!3\;\longrightarrow\;\text{pointwise }1\!\times\!1\;\longrightarrow\;y.$$

The values need not match because the weights are independently initialized. The **contract** to verify is the shape.

In [3]:
standard = nn.Conv2d(c_in, c_out, kernel_size=k, padding=1, bias=False)
separable = nn.Sequential(
    nn.Conv2d(c_in, c_in, kernel_size=k, padding=1, groups=c_in, bias=False),
    nn.Conv2d(c_in, c_out, kernel_size=1, bias=False),
)

x = torch.randn(2, c_in, H, W)
y_standard = standard(x)
y_depthwise = separable[0](x)
y_separable = separable(x)

print("input       ", tuple(x.shape))
print("depthwise   ", tuple(y_depthwise.shape))
print("standard out", tuple(y_standard.shape))
print("separable out", tuple(y_separable.shape))
print("module parameters:", n_params(standard), n_params(separable))

assert y_standard.shape == y_separable.shape == (2, 128, 28, 28)
assert y_depthwise.shape == (2, 64, 28, 28)
assert n_params(standard) == standard_weights
assert n_params(separable) == separable_weights

input        (2, 64, 28, 28)
depthwise    (2, 64, 28, 28)
standard out (2, 128, 28, 28)
separable out (2, 128, 28, 28)
module parameters: 73728 8768


### Explain

Write one sentence for each blank before revealing your notes:

1. The depthwise node changes ______ but not ______.  
2. The pointwise node changes ______ by mixing ______.  
3. The factorized block is cheaper because it avoids learning a separate $3\times3$ kernel for every ______.

Expected ideas: the depthwise node changes spatial responses but not channel count; the pointwise node changes channel count by mixing channels; the avoided object is every input–output channel pair.

## 2 · Preserve an identity path

The lecture used the scalar residual block

$$y=x+F(x), \qquad F(x)=\alpha x^2.$$

**Predict before running:** at $x=2$ and $\alpha=0.1$, what are $F(x)$, $y$, and $dy/dx$? What happens when $\alpha=0$?

In [4]:
def residual_scalar(x, alpha):
    return x + alpha * x.square()

x_scalar = torch.tensor(2.0, requires_grad=True)
alpha = torch.tensor(0.1)
y_scalar = residual_scalar(x_scalar, alpha)
y_scalar.backward()

print(f"x={x_scalar.item():.1f}")
print(f"F(x)={(alpha * x_scalar.detach().square()).item():.1f}")
print(f"y=x+F(x)={y_scalar.item():.1f}")
print(f"dy/dx={x_scalar.grad.item():.1f}")

assert math.isclose(y_scalar.item(), 2.4, rel_tol=1e-6)
assert math.isclose(x_scalar.grad.item(), 1.4, rel_tol=1e-6)

x=2.0
F(x)=0.4
y=x+F(x)=2.4
dy/dx=1.4


In [5]:
x_identity = torch.tensor([-2.0, 0.5, 3.0], requires_grad=True)
alpha_zero = torch.tensor(0.0)
y_identity = residual_scalar(x_identity, alpha_zero)
y_identity.sum().backward()

print("input :", x_identity.detach())
print("output:", y_identity.detach())
print("gradient through the block:", x_identity.grad)

assert torch.equal(y_identity.detach(), x_identity.detach())
assert torch.equal(x_identity.grad, torch.ones_like(x_identity))

input : tensor([-2.0000,  0.5000,  3.0000])
output: tensor([-2.0000,  0.5000,  3.0000])
gradient through the block: tensor([1., 1., 1.])


### Inspect the result

At $\alpha=0$, the residual branch contributes zero, so the forward map is the identity. The derivative is still one because the shortcut remains. This is the concrete meaning of “the identity path survives.”

**Misconception check:** the shortcut does not stop learning. When $\alpha\neq0$, the output and gradient both include the learned residual term.

## 3 · Reuse a backbone, replace the head

The lecture's held case uses a backbone feature $h\in\mathbb{R}^{512}$ and $K=6$ new classes, so the replacement head has $6\cdot512+6=3{,}078$ parameters. We reproduce that exact interface with a small local CNN containing BatchNorm.

> This backbone is randomly initialized. It is a fast, inspectable stand-in for checking shapes, gradients, parameter updates, and module state; it cannot demonstrate that transferred features are useful. Transfer quality requires a genuinely pretrained backbone and held-out validation data.

**Predict before running:** after setting `requires_grad=False` on every backbone parameter, which parameters can change? If the frozen backbone remains in training mode, can a BatchNorm running-mean buffer still change?

In [6]:
class TinyBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1),
            nn.BatchNorm2d(8),
            nn.ReLU(),
        )
        self.late = nn.Sequential(
            nn.Conv2d(8, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(16, 512),
        )

    def forward(self, x):
        return self.late(self.stem(x))


class TransferClassifier(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.backbone = TinyBackbone()
        self.head = nn.Linear(512, n_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


model = TransferClassifier(n_classes=6)
for parameter in model.backbone.parameters():
    parameter.requires_grad = False

lecture_backbone_parameters = 11_000_000
expected_head_parameters = 6 * 512 + 6
probe_fraction = expected_head_parameters / (lecture_backbone_parameters + expected_head_parameters)

features = model.backbone(torch.randn(2, 3, 16, 16))
logits = model.head(features)
print("feature shape        :", tuple(features.shape))
print("logit shape          :", tuple(logits.shape))
print(f"new head parameters : 6×512 + 6 = {n_params(model.head):,}")
print(f"all parameters      : {n_params(model):,}")
print(f"trainable parameters: {n_params(model, trainable_only=True):,}")
print(f"lecture probe share : {100 * probe_fraction:.3f}% of 11M + head")

assert features.shape == (2, 512)
assert logits.shape == (2, 6)
assert n_params(model.head) == expected_head_parameters == 3_078
assert n_params(model, trainable_only=True) == 3_078
assert math.isclose(100 * probe_fraction, 0.027974, rel_tol=1e-5)

feature shape        : (2, 512)
logit shape          : (2, 6)
new head parameters : 6×512 + 6 = 3,078
all parameters      : 13,222
trainable parameters: 3,078
lecture probe share : 0.028% of 11M + head


### Freezing weights is not freezing module state

`requires_grad=False` prevents parameter gradients and optimizer updates. It does **not** switch BatchNorm or Dropout to inference behaviour: `train()`/`eval()` controls that separate contract.

The next cell first shows that a frozen BatchNorm buffer can move in training mode. It then restores the state and performs a proper linear-probe step with the backbone in evaluation mode and the head in training mode. **Predict:** which of the four reported changes should be exactly zero?

In [7]:
images = torch.randn(12, 3, 16, 16)
labels = torch.tensor([0, 1, 2, 3, 4, 5] * 2)

# A parameter freeze alone does not freeze BatchNorm buffers.
clean_backbone_state = copy.deepcopy(model.backbone.state_dict())
stem_bn = model.backbone.stem[1]
running_mean_before = stem_bn.running_mean.detach().clone()
model.backbone.train()
with torch.no_grad():
    _ = model.backbone(images)
train_mode_bn_change = (stem_bn.running_mean - running_mean_before).abs().max().item()

# Restore state, then use the full linear-probe contract.
model.backbone.load_state_dict(clean_backbone_state)
model.head.train()
model.backbone.eval()
backbone_before = {name: p.detach().clone() for name, p in model.backbone.named_parameters()}
head_before = {name: p.detach().clone() for name, p in model.head.named_parameters()}
running_mean_before = stem_bn.running_mean.detach().clone()

probe_optimizer = torch.optim.SGD(model.head.parameters(), lr=0.2)
criterion = nn.CrossEntropyLoss()

probe_optimizer.zero_grad()
logits = model(images)
loss = criterion(logits, labels)
loss.backward()
probe_optimizer.step()

backbone_change = max(
    (p.detach() - backbone_before[name]).abs().max().item()
    for name, p in model.backbone.named_parameters()
)
head_change = max(
    (p.detach() - head_before[name]).abs().max().item()
    for name, p in model.head.named_parameters()
)
eval_mode_bn_change = (stem_bn.running_mean - running_mean_before).abs().max().item()
backbone_grad_is_none = all(p.grad is None for p in model.backbone.parameters())

print(f"frozen BN buffer change in train mode: {train_mode_bn_change:.6f}")
print(f"linear-probe loss                    : {loss.item():.4f}")
print(f"backbone parameter change            : {backbone_change:.6f}")
print(f"backbone BN change in eval mode      : {eval_mode_bn_change:.6f}")
print(f"head parameter change                : {head_change:.6f}")
print(f"backbone gradients are None          : {backbone_grad_is_none}")

assert train_mode_bn_change > 0.0
assert backbone_change == 0.0
assert eval_mode_bn_change == 0.0
assert head_change > 0.0
assert backbone_grad_is_none

frozen BN buffer change in train mode: 0.011999
linear-probe loss                    : 1.7955
backbone parameter change            : 0.000000
backbone BN change in eval mode      : 0.000000
head parameter change                : 0.004505
backbone gradients are None          : True


### Explain the update

- The frozen backbone still performed a forward computation; “frozen” means **not updated**, not “unused.”
- Its parameters had `requires_grad=False`, so no parameter gradients were stored and the optimizer contained only the head.
- BatchNorm running statistics are buffers, not parameters. They moved when the frozen backbone was in training mode and stayed fixed after `backbone.eval()`. Dropout would likewise remain stochastic in training mode.
- The trainable head received gradients and changed after the optimizer step. In a loop, call `model.train()` and then `model.backbone.eval()` on every linear-probe epoch, because `model.train()` recursively reactivates descendants.

This verifies mechanics only. It does not show that a random backbone transfers useful features.

## 4 · Let validation control progressive unfreezing

Do not unfreeze because a fixed epoch number elapsed, and never consult the test set. First train the probe while monitoring a held-out validation split. If training and validation both plateau at a poor level with only a small gap, the frozen representation may be the bottleneck; then unfreeze one late block and revalidate. A widening train–validation gap argues for more regularization or data, not more trainable capacity.

The trace below is an **illustrative control input**, not a result produced by the synthetic tensors above. **Predict before running:** will its small validation improvement and small train–validation gap keep the probe frozen or trigger one late-block unfreeze?

In [8]:
probe_train_loss = np.array([1.70, 1.56, 1.50, 1.49])
probe_val_loss = np.array([1.74, 1.59, 1.520, 1.519])
min_delta = 0.005
max_small_gap = 0.05
target_val_loss = 1.00  # a task-specific target chosen before inspecting the test set

recent_val_improvement = probe_val_loss[-2] - probe_val_loss[-1]
final_generalization_gap = probe_val_loss[-1] - probe_train_loss[-1]
still_below_target = probe_val_loss[-1] > target_val_loss
unfreeze_late = (
    recent_val_improvement < min_delta
    and final_generalization_gap < max_small_gap
    and still_below_target
)
decision = "unfreeze one late block" if unfreeze_late else "keep the linear probe"

print(f"recent validation improvement: {recent_val_improvement:.3f}")
print(f"final train–validation gap   : {final_generalization_gap:.3f}")
print(f"validation target met        : {not still_below_target}")
print("validation-controlled decision:", decision)

assert unfreeze_late
for parameter in model.backbone.late.parameters():
    parameter.requires_grad = True

late_parameters = [p for p in model.backbone.late.parameters() if p.requires_grad]
head_parameters = [p for p in model.head.parameters() if p.requires_grad]
fine_tune_optimizer = torch.optim.SGD(
    [
        {"params": late_parameters, "lr": 0.01, "name": "late_backbone"},
        {"params": head_parameters, "lr": 0.10, "name": "head"},
    ]
)
print("optimizer groups:", [(g["name"], g["lr"]) for g in fine_tune_optimizer.param_groups])

# One bounded mechanics step: frozen stem fixed; late block and head update.
model.train()
model.backbone.stem.eval()
stem_before = {name: p.detach().clone() for name, p in model.backbone.stem.named_parameters()}
late_before = {name: p.detach().clone() for name, p in model.backbone.late.named_parameters()}
head_before = {name: p.detach().clone() for name, p in model.head.named_parameters()}

fine_tune_optimizer.zero_grad()
fine_tune_loss = criterion(model(images), labels)
fine_tune_loss.backward()
fine_tune_optimizer.step()

def largest_parameter_change(module, before):
    return max(
        (parameter.detach() - before[name]).abs().max().item()
        for name, parameter in module.named_parameters()
    )

stem_change = largest_parameter_change(model.backbone.stem, stem_before)
late_change = largest_parameter_change(model.backbone.late, late_before)
head_change = largest_parameter_change(model.head, head_before)
print(f"frozen stem change : {stem_change:.6f}")
print(f"late block change  : {late_change:.6f}")
print(f"head change        : {head_change:.6f}")

assert [group["lr"] for group in fine_tune_optimizer.param_groups] == [0.01, 0.10]
assert stem_change == 0.0
assert late_change > 0.0
assert head_change > 0.0
assert all(p.grad is None for p in model.backbone.stem.parameters())

recent validation improvement: 0.001
final train–validation gap   : 0.029
validation target met        : False
validation-controlled decision: unfreeze one late block
optimizer groups: [('late_backbone', 0.01), ('head', 0.1)]
frozen stem change : 0.000000
late block change  : 0.000113
head change        : 0.001796


## 5 · Synthesis

Complete these explanations without architecture names:

1. Two stacked $3\times3$ convolutions can replace one $5\times5$ because ______.  
2. A $1\times1$ bottleneck saves compute by reducing ______ before ______.  
3. A residual shortcut makes ______ behaviour easy to represent.  
4. A depthwise-separable block factors ______ from ______.  
5. A linear probe tests whether pretrained ______ are already useful for new ______.  
6. Freezing parameters does not freeze BatchNorm ______ or Dropout ______; that requires the right module mode.  
7. The test set remains sealed while the ______ split controls whether a late block is unfrozen.

If you can answer these precisely—and reproduce the counts above—you understand the design reasoning behind the blocks, not merely their names.